# 🧠 Colab A — 3-Layer Neural Network: NumPy Only + tf.einsum
**CMPE-256 | Non-linear Regression from Scratch**

- ✅ 3-layer deep neural network (Input→64→32→16→Output)
- ✅ Manual forward pass, backpropagation, chain rule
- ✅ `tf.einsum` instead of matrix multiply (as required)
- ✅ 3-variable non-linear equation: `y = 2x₁² + 3x₂x₃ + sin(x₁x₂) + 0.5x₃²`
- ✅ 4D visualization using PCA (scikit-learn)

In [ ]:
# ── Section 0: Install & Imports ──────────────────────────────────────────
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)
print('NumPy:', np.__version__, '| TF:', tf.__version__)

## 📊 Section 1 — Synthetic Data Generation & 4D Visualization

In [ ]:
# ── Section 1: Generate 3-variable non-linear synthetic data ──────────────
N = 1000
x1 = np.random.uniform(-2, 2, (N, 1))
x2 = np.random.uniform(-2, 2, (N, 1))
x3 = np.random.uniform(-2, 2, (N, 1))

# Non-linear equation with 3 variables
y = (2*x1**2 + 3*x2*x3 + np.sin(x1*x2) + 0.5*x3**2
     + np.random.normal(0, 0.1, (N, 1)))

X = np.hstack([x1, x2, x3])   # shape (1000, 3)
print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'y range: [{y.min():.2f}, {y.max():.2f}]')

In [ ]:
# ── Section 1b: 4D Visualization via PCA ─────────────────────────────────
# PCA reduces 3 input features → 2 principal components for plotting
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_}')

fig = plt.figure(figsize=(12, 5))

# 3D scatter: PC1, PC2, y  (color = y value = 4th dimension)
ax1 = fig.add_subplot(121, projection='3d')
sc = ax1.scatter(X_pca[:,0], X_pca[:,1], y[:,0],
                 c=y[:,0], cmap='coolwarm', alpha=0.6, s=10)
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2'); ax1.set_zlabel('y')
ax1.set_title('4D View: PC1 | PC2 | y | color=y')
plt.colorbar(sc, ax=ax1, shrink=0.5)

# 2D projection: PC1 vs y colored by x3
ax2 = fig.add_subplot(122)
sc2 = ax2.scatter(X_pca[:,0], y[:,0], c=x3[:,0], cmap='viridis', alpha=0.5, s=10)
ax2.set_xlabel('PC1'); ax2.set_ylabel('y')
ax2.set_title('PC1 vs y (color = x3 value)')
plt.colorbar(sc2, ax=ax2)
plt.tight_layout()
plt.savefig('4d_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('4D plot saved as 4d_plot.png')

## 🏗️ Section 2 — Data Preprocessing & Train/Test Split

In [ ]:
# ── Section 2: Normalize and split ──────────────────────────────────────
# Normalize inputs
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / X_std

y_mean, y_std = y.mean(), y.std()
y_norm = (y - y_mean) / y_std

# 80/20 train/test split
split = int(0.8 * N)
X_train, X_test = X_norm[:split], X_norm[split:]
y_train, y_test = y_norm[:split], y_norm[split:]
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## ⚙️ Section 3 — Weight Initialization (He Init)

In [ ]:
# ── Section 3: He initialization ─────────────────────────────────────────
def init_weights(layer_dims):
    """He initialization for ReLU networks: scale = sqrt(2/fan_in)"""
    params = {}
    for l in range(1, len(layer_dims)):
        fan_in = layer_dims[l-1]
        scale = np.sqrt(2.0 / fan_in)
        params[f'W{l}'] = np.random.randn(layer_dims[l-1], layer_dims[l]) * scale
        params[f'b{l}'] = np.zeros((1, layer_dims[l]))
    return params

# Architecture: 3 → 64 → 32 → 16 → 1
layer_dims = [3, 64, 32, 16, 1]
params = init_weights(layer_dims)
for k, v in params.items():
    print(f'  {k}: {v.shape}')

## ➡️ Section 4 — Forward Pass using tf.einsum

In [ ]:
# ── Section 4: Forward pass — USES tf.einsum (assignment requirement) ──────
def relu(z):    return np.maximum(0, z)
def tanh(z):    return np.tanh(z)
def relu_grad(z): return (z > 0).astype(float)   # derivative of ReLU
def tanh_grad(z): return 1 - np.tanh(z)**2        # derivative of Tanh

def forward_pass(X, params):
    """
    Forward pass using tf.einsum for all matrix multiplications.
    einsum 'bi,ij->bj' = matrix multiply of (batch x in) @ (in x out) → (batch x out)
    """
    cache = {'A0': X}
    X_tf = tf.constant(X, dtype=tf.float32)

    # Hidden Layer 1: ReLU  (3 → 64)
    W1_tf = tf.constant(params['W1'], dtype=tf.float32)
    Z1 = tf.einsum('bi,ij->bj', X_tf, W1_tf).numpy() + params['b1']
    A1 = relu(Z1)
    cache.update({'Z1': Z1, 'A1': A1})

    # Hidden Layer 2: ReLU  (64 → 32)
    W2_tf = tf.constant(params['W2'], dtype=tf.float32)
    A1_tf = tf.constant(A1, dtype=tf.float32)
    Z2 = tf.einsum('bi,ij->bj', A1_tf, W2_tf).numpy() + params['b2']
    A2 = relu(Z2)
    cache.update({'Z2': Z2, 'A2': A2})

    # Hidden Layer 3: Tanh  (32 → 16)
    W3_tf = tf.constant(params['W3'], dtype=tf.float32)
    A2_tf = tf.constant(A2, dtype=tf.float32)
    Z3 = tf.einsum('bi,ij->bj', A2_tf, W3_tf).numpy() + params['b3']
    A3 = tanh(Z3)
    cache.update({'Z3': Z3, 'A3': A3})

    # Output Layer: Linear  (16 → 1)
    W4_tf = tf.constant(params['W4'], dtype=tf.float32)
    A3_tf = tf.constant(A3, dtype=tf.float32)
    Z4 = tf.einsum('bi,ij->bj', A3_tf, W4_tf).numpy() + params['b4']
    cache.update({'Z4': Z4, 'A4': Z4})   # linear output, A4 = Z4

    return Z4, cache

# Quick test
out, cache = forward_pass(X_train[:4], params)
print('Forward pass output shape:', out.shape)
print('Cache keys:', list(cache.keys()))

## 🔙 Section 5 — Manual Backpropagation (Chain Rule)

In [ ]:
# ── Section 5: Manual Backprop ── Chain Rule for all 4 layers ─────────────
def backward_pass(y_true, cache, params):
    """
    Chain rule gradient propagation through 4 layers.
    Layer order (backward): Output → Layer3 → Layer2 → Layer1
    """
    n = y_true.shape[0]
    grads = {}

    # ── Output layer: Linear activation, MSE loss ──
    # dL/dZ4 = (2/n) * (A4 - y)  [MSE gradient]
    dZ4 = (2.0 / n) * (cache['A4'] - y_true)      # (n, 1)
    grads['dW4'] = cache['A3'].T @ dZ4             # (16, 1)
    grads['db4'] = dZ4.mean(axis=0, keepdims=True) # (1, 1)
    dA3 = dZ4 @ params['W4'].T                     # (n, 16) — propagate back

    # ── Layer 3: Tanh activation ──
    # dL/dZ3 = dL/dA3 * d(tanh)/dZ3 = dA3 * (1 - tanh²(Z3))
    dZ3 = dA3 * tanh_grad(cache['Z3'])             # (n, 16)
    grads['dW3'] = cache['A2'].T @ dZ3             # (32, 16)
    grads['db3'] = dZ3.mean(axis=0, keepdims=True) # (1, 16)
    dA2 = dZ3 @ params['W3'].T                     # (n, 32)

    # ── Layer 2: ReLU activation ──
    # dL/dZ2 = dL/dA2 * d(relu)/dZ2 = dA2 * (Z2 > 0)
    dZ2 = dA2 * relu_grad(cache['Z2'])             # (n, 32)
    grads['dW2'] = cache['A1'].T @ dZ2             # (64, 32)
    grads['db2'] = dZ2.mean(axis=0, keepdims=True) # (1, 32)
    dA1 = dZ2 @ params['W2'].T                     # (n, 64)

    # ── Layer 1: ReLU activation ──
    # dL/dZ1 = dL/dA1 * d(relu)/dZ1 = dA1 * (Z1 > 0)
    dZ1 = dA1 * relu_grad(cache['Z1'])             # (n, 64)
    grads['dW1'] = cache['A0'].T @ dZ1             # (3, 64)
    grads['db1'] = dZ1.mean(axis=0, keepdims=True) # (1, 64)

    return grads

# Gradient shape check
out, cache = forward_pass(X_train[:16], params)
grads = backward_pass(y_train[:16], cache, params)
for k, v in grads.items():
    wk = k.replace('d','');  wk = wk if wk in params else None
    match = params[wk].shape == v.shape if wk else '—'
    print(f'  {k}: {v.shape}  shapes_match={match}')

## 📦 Section 6 — Adam Optimizer from Scratch

In [ ]:
# ── Section 6: Adam optimizer ─────────────────────────────────────────────
class AdamOptimizer:
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m, self.v, self.t = {}, {}, 0

    def update(self, params, grads):
        self.t += 1
        for key in ['W1','b1','W2','b2','W3','b3','W4','b4']:
            g = grads[f'd{key}']
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            # 1st moment (momentum)
            self.m[key] = self.beta1*self.m[key] + (1-self.beta1)*g
            # 2nd moment (adaptive LR)
            self.v[key] = self.beta2*self.v[key] + (1-self.beta2)*(g**2)
            # Bias correction
            m_hat = self.m[key] / (1 - self.beta1**self.t)
            v_hat = self.v[key] / (1 - self.beta2**self.t)
            # Parameter update
            params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return params

optimizer = AdamOptimizer(lr=1e-3)
print('Adam optimizer initialized')

## 🏋️ Section 7 — Training Loop

In [ ]:
# ── Section 7: Training loop — 1000 epochs ────────────────────────────────
EPOCHS = 1000
BATCH  = 64
history = []
params  = init_weights([3, 64, 32, 16, 1])   # fresh init
optimizer = AdamOptimizer(lr=1e-3)

for epoch in range(EPOCHS):
    # Mini-batch shuffle
    idx = np.random.permutation(X_train.shape[0])
    Xb, Yb = X_train[idx[:BATCH]], y_train[idx[:BATCH]]

    # Forward → Loss → Backward → Update
    y_pred, cache = forward_pass(Xb, params)
    loss = np.mean((y_pred - Yb)**2)
    grads = backward_pass(Yb, cache, params)
    params = optimizer.update(params, grads)
    history.append(float(loss))

    if (epoch+1) % 100 == 0:
        val_pred, _ = forward_pass(X_test, params)
        val_loss = np.mean((val_pred - y_test)**2)
        print(f'Epoch {epoch+1:4d} | Train Loss: {loss:.5f} | Val Loss: {val_loss:.5f}')

## 📈 Section 8 — Results: Loss Curve & Predictions

In [ ]:
# ── Section 8a: Loss curve ───────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.semilogy(history, color='steelblue', linewidth=1.5, label='Train Loss (MSE)')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss (log scale)')
plt.title('Colab A — NumPy Neural Net Training Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Final train loss: {history[-1]:.6f}')

In [ ]:
# ── Section 8b: Predicted vs Actual ─────────────────────────────────────
y_pred_test, _ = forward_pass(X_test, params)
# Denormalize
y_pred_denorm = y_pred_test * y_std + y_mean
y_test_denorm  = y_test  * y_std + y_mean

plt.figure(figsize=(6,6))
plt.scatter(y_test_denorm, y_pred_denorm, alpha=0.4, s=15, color='steelblue')
mn, mx = y_test_denorm.min(), y_test_denorm.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual y'); plt.ylabel('Predicted y')
plt.title('Colab A — Predicted vs Actual (Test Set)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

mse_test = np.mean((y_pred_denorm - y_test_denorm)**2)
mae_test = np.mean(np.abs(y_pred_denorm - y_test_denorm))
print(f'Test MSE: {mse_test:.4f} | Test MAE: {mae_test:.4f}')